In [ ]:
!pip install konlpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 56.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 495.9/495.9 kB 39.7 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import re
from konlpy.tag import Okt,Mecab
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score,f1_score
from lightgbm import LGBMClassifier

In [ ]:
train = pd.read_csv('/content/drive/MyDrive/data/뉴스토픽 분류/train_data.csv')
test = pd.read_csv('/content/drive/MyDrive/data/뉴스토픽 분류/test_data.csv')
sample_submission = pd.read_csv('/content/drive/MyDrive/data/뉴스토픽 분류/sample_submission.csv')

## 데이터 전처리

In [ ]:
!apt-get update
!apt-get install -y openjdk-11-jdk

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 257 kB in 2s (155 kB/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provid

In [ ]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["PATH"] += os.pathsep + os.path.join(os.environ["JAVA_HOME"], "bin")

In [ ]:
# 형태소 분석기(Okt) 불러오기
okt=Okt()

In [ ]:
# 조사, 어미, 구두점 제거
def func(text):
    clean = []
    for word in okt.pos(text, stem=True): #어간 추출
        if word[1] not in ['Josa', 'Eomi', 'Punctuation']: #조사, 어미, 구두점 제외
            clean.append(word[0])


    return " ".join(clean)

train['title'] = train['title'].apply(lambda x : func(x))

In [ ]:
# tf-idf를 이용한 벡터화
def split(text):
    tokens_ko = text.split()
    return tokens_ko

tfidf_vect = TfidfVectorizer(tokenizer=split, ngram_range=(1,2), max_features=8000)
tfidf_vect.fit(train['title'])
tfidf_matrix_train = tfidf_vect.transform(train['title'])

/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [ ]:
# train/valid 데이터 셋 나누기.
def split_dataset(tfidf,df):
    X_data = tfidf
    y_data = df['topic_idx']

    # stratify=y_data Stratified 기반 분할, train 데이터의 30%를 평가 데이터 셋으로 사용. (70% 데이터 학습에 사용)
    X_train, X_test, y_train, y_test = \
    train_test_split(X_data, y_data, test_size=0.3, random_state=42, stratify=y_data)


    return (X_train, X_test, y_train, y_test)

X_train, X_test, y_train, y_test = split_dataset(tfidf_matrix_train,train)

## 모델 학습

In [ ]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=42)

In [ ]:
# ===============================
# 검증 세트 성능 확인
# ===============================
val_pred = rf.predict(X_test)
print("Accuracy:", accuracy_score(y_test, val_pred))
print("Macro F1:", f1_score(y_test, val_pred, average='macro'))

Accuracy: 0.797400890705994
Macro F1: 0.7995074227810507


## 스태킹

In [ ]:
import pandas as pd
import numpy as np
import re, os, time
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, f1_score, make_scorer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, StackingClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression

In [ ]:
# 후보 base/meta 세트 (소형)
base_model_sets = {
    'rf_et': [
        ('rf', RandomForestClassifier(n_estimators=150, max_depth=25, random_state=42, n_jobs=1)),
        ('et', ExtraTreesClassifier(n_estimators=150, max_depth=25, random_state=43, n_jobs=1))
    ],
    'rf_lgbm': [
        ('rf', RandomForestClassifier(n_estimators=150, max_depth=20, random_state=42, n_jobs=1)),
        ('lgb', LGBMClassifier(n_estimators=100, learning_rate=0.1, random_state=44, n_jobs=1))
    ]
}

meta_models = {
    'logreg': LogisticRegression(max_iter=1000),
    'lgbm': LGBMClassifier(n_estimators=150, learning_rate=0.05, random_state=42, n_jobs=1)
}

f1_macro = make_scorer(f1_score, average='macro')

results = []

print("스태킹 조합별 교차검증 시작...\n")
for set_name, base_models in base_model_sets.items():
    for meta_name, meta_model in meta_models.items():
        start = time.time()
        print(f"▶ Base={set_name} | Meta={meta_name}")
        stack_model = StackingClassifier(
            estimators=base_models,
            final_estimator=meta_model,
            cv=2,
            n_jobs=1,
            passthrough=True
        )
        pipe = Pipeline([
            ('select', SelectKBest(chi2, k=5000)),
            ('stack', stack_model)
        ])
        scores = cross_val_score(pipe, X_train, y_train, cv=2, scoring=f1_macro, n_jobs=1)
        mean_f1 = np.mean(scores)
        print(f"F1 Macro: {mean_f1:.4f} | Time: {time.time()-start:.1f}s\n")
        results.append((set_name, meta_name, mean_f1))

result_df = pd.DataFrame(results, columns=['BaseSet', 'MetaModel', 'F1_Macro'])
print("\n최적 조합 결과:")
print(result_df.sort_values(by='F1_Macro', ascending=False))

best_combo = result_df.sort_values(by='F1_Macro', ascending=False).iloc[0]
best_base = best_combo['BaseSet']
best_meta = best_combo['MetaModel']

print(f"\n최적 조합: Base={best_base}, Meta={best_meta}")

스태킹 조합별 교차검증 시작...

▶ Base=rf_et | Meta=logreg
F1 Macro: 0.8176 | Time: 54.3s

▶ Base=rf_et | Meta=lgbm
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.280314 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 34486
[LightGBM] [Info] Number of data points in the train set: 15978, number of used features: 1476
[LightGBM] [Info] Start training from score -2.247076
[LightGBM] [Info] Start training from score -1.992806
[LightGBM] [Info] Start training from score -1.824975
[LightGBM] [Info] Start training from score -2.040770
[LightGBM] [Info] Start training from score -1.789134
[LightGBM] [Info] Start training from score -1.884969
[LightGBM] [Info] Start training from score -1.911281


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.280983 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 34900
[LightGBM] [Info] Number of data points in the train set: 15979, number of used features: 1479
[LightGBM] [Info] Start training from score -2.247731
[LightGBM] [Info] Start training from score -1.993328
[LightGBM] [Info] Start training from score -1.824649
[LightGBM] [Info] Start training from score -2.040351
[LightGBM] [Info] Start training from score -1.789197
[LightGBM] [Info] Start training from score -1.884619
[LightGBM] [Info] Start training from score -1.911343


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


F1 Macro: 0.7687 | Time: 86.9s

▶ Base=rf_lgbm | Meta=logreg
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.174669 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 30916
[LightGBM] [Info] Number of data points in the train set: 15978, number of used features: 1462
[LightGBM] [Info] Start training from score -2.247076
[LightGBM] [Info] Start training from score -1.992806
[LightGBM] [Info] Start training from score -1.824975
[LightGBM] [Info] Start training from score -2.040770
[LightGBM] [Info] Start training from score -1.789134
[LightGBM] [Info] Start training from score -1.884969
[LightGBM] [Info] Start training from score -1.911281
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.043394 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `f

...

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.044886 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 12554
[LightGBM] [Info] Number of data points in the train set: 7990, number of used features: 722
[LightGBM] [Info] Start training from score -2.247794
[LightGBM] [Info] Start training from score -1.993850
[LightGBM] [Info] Start training from score -1.825100
[LightGBM] [Info] Start training from score -2.039932
[LightGBM] [Info] Start training from score -1.789259
[LightGBM] [Info] Start training from score -1.884270
[LightGBM] [Info] Start training from score -1.910983


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.285053 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 34900
[LightGBM] [Info] Number of data points in the train set: 15979, number of used features: 1479
[LightGBM] [Info] Start training from score -2.247731
[LightGBM] [Info] Start training from score -1.993328
[LightGBM] [Info] Start training from score -1.824649
[LightGBM] [Info] Start training from score -2.040351
[LightGBM] [Info] Start training from score -1.789197
[LightGBM] [Info] Start training from score -1.884619
[LightGBM] [Info] Start training from score -1.911343


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


F1 Macro: 0.7783 | Time: 111.1s


최적 조합 결과:
   BaseSet MetaModel  F1_Macro
0    rf_et    logreg  0.817587
2  rf_lgbm    logreg  0.813724
3  rf_lgbm      lgbm  0.778340
1    rf_et      lgbm  0.768721

최적 조합: Base=rf_et, Meta=logreg


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression

# 최적 조합 설정
base_models = [
    ('rf', RandomForestClassifier(n_estimators=150, max_depth=25, random_state=42, n_jobs=1)),
    ('et', ExtraTreesClassifier(n_estimators=150, max_depth=25, random_state=43, n_jobs=1))
]
meta_model = LogisticRegression(max_iter=1000)

# 스태킹 모델 정의
final_stack = StackingClassifier(
    estimators=base_models,
    final_estimator=meta_model,
    cv=3,
    n_jobs=1,
    passthrough=True
)

# 최종 파이프라인
final_pipe = Pipeline([
    ('select', SelectKBest(chi2, k=5000)),
    ('stack', final_stack)
])

# 전체 학습 데이터로 재학습
print("최적 스태킹 모델 학습 시작...")
final_pipe.fit(tfidf_matrix_train, train['topic_idx'])

# 테스트 데이터 전처리
test['title'] = test['title'].apply(lambda x : func(x))
tfidf_matrix_test = tfidf_vect.transform(test['title'])

# 테스트 데이터 예측
pred = final_pipe.predict(tfidf_matrix_test)

# 제출 파일 생성
sample_submission['topic_idx'] = pred
output_path = '/content/drive/MyDrive/data/뉴스토픽_최종_rf_et_logreg.csv'
sample_submission.to_csv(output_path, index=False)
print(f"\n 최적 스태킹 모델 예측 완료! 제출 파일 저장: {output_path}")

최적 스태킹 모델 학습 시작...

 최적 스태킹 모델 예측 완료! 제출 파일 저장: /content/drive/MyDrive/data/뉴스토픽_최종_rf_et_logreg.csv
